# Analysis of Shots taken by Kobe Bryant

Kobe Bryant had been one of the greatest NBA players of all time and my favorite player since I started watching NBA(May he rest in peace). To better understand his game and legacy, I want to utilize the knowledge I learned about data science to analyze all the shots Kobe Bryant had taken in his career. 

![](https://media3.s-nbcnews.com/i/newscms/2020_04/3203206/200126-kobe-bryant-cs-551p_94a33c06ae6c4bbebaf59985de472c42.jpg)

Also I want to thank [Savvas Tjortjoglou](http://savvastjortjoglou.com/) and [JP Hwang](https://www.jphwang.com/) for the tutorial of NBA shot charts and [nba_api](https://github.com/swar/nba_api) for allowing me access Kobe's shotlog from [NBA](https://www.nba.com/).

Import Python Packages

In [ ]:
#to analyze data
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
from matplotlib.patches import Circle, Rectangle, Arc
from matplotlib.colors import Normalize
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import plotly.offline as offline
from IPython.display import Image

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

### Load Kobe's shotlog

Clean the data by delete the columns that we are not gonna analyze like Game ID, Team ID, Player ID, etc. 

In [ ]:
kobe_shotlog = pd.read_csv("../input/kobe-bryants-shotlog/Kobe_Shotlog.csv")
kobe_shotlog.drop(['GRID_TYPE', 'GAME_EVENT_ID', 'PLAYER_ID', 'TEAM_ID', 'MINUTES_REMAINING','SECONDS_REMAINING', 'SHOT_ATTEMPTED_FLAG', 'GAME_DATE', 'HTM', 'VTM'], axis=1, inplace=True)
kobe_shotlog

The dataframe/table above contains all 30697 shots Kobe Bryant had taken in his career and their corresponding information like type of the shot, whether he made it or not, location of the shot on the court, distance from the hoop, etc. 

Note that shots taken in the Pre-season, All-Star games, Olympics games are not included. 

## Analysis of the types of shots

Group the table by Season and Shot Type, then count each types of shots(2PT or 3PT) and then calculate the percentages of each correspond shot type in each season.

In [ ]:
#Analyze type shots Kobe takes through his career
shot_type = kobe_shotlog.groupby('Season')['SHOT_TYPE'].value_counts().to_frame()
shot_type.rename(columns={'SHOT_TYPE':'Counts'}, inplace=True)
shot_type.reset_index(inplace=True)

#Get all seasons Kobe played
seasons = shot_type.Season.value_counts().index.sort_values()
#Calculate the percentage of each shot type in Kobe's career
shot_type ['Percentage'] = np.nan
for season in seasons:
    two_point = shot_type.loc[(shot_type['Season']==season) & (shot_type['SHOT_TYPE']=='2PT Field Goal')].Counts.to_numpy()[0]
    three_point = shot_type.loc[(shot_type['Season']==season) & (shot_type['SHOT_TYPE']=='3PT Field Goal')].Counts.to_numpy()[0]
    two_percentage = round(two_point/(two_point + three_point),3) * 100
    three_percentage = round(three_point/(two_point + three_point),3) * 100
    shot_type.loc[(shot_type['Season']==season) & (shot_type['SHOT_TYPE']=='2PT Field Goal'),'Percentage']=two_percentage
    shot_type.loc[(shot_type['Season']==season) & (shot_type['SHOT_TYPE']=='3PT Field Goal'),'Percentage']=three_percentage
    
shot_type

One thing we need to notice is that in the 2013-14 season, Kobe only took a total of 73 shots. After an career ending injury(Torn Achilles) in the previous season, Kobe only played 6 games in that 2013-14 season. Because of the small quantity of data points, the analysis of data in this season may be outliers and does not have statistical significance. 

### Visualize the data with bar graph

In [ ]:
#Graph shot type percentage
barchart = px.bar(data_frame=shot_type,x="Percentage",y="Season",color="SHOT_TYPE",color_discrete_sequence=['#fdb927','#552583'], orientation="h",width=900, height=700,opacity=0.85)
barchart.update_yaxes(type='category')
barchart.show()

As we can see on the graph, except for Kobe's first season and last season when over one third of Kobe's shots are 3-pointers, the majority of Kobe's shots are 2-pointers and they can be dunks, pull-up jumpers, fadeaways and so on. 

In his last season, the 2015-2016 season, the offense of the league was evolving led by Stephen Curry and Golden State Warriors where teams started to shoot much more 3-pointers. At the same time, Kobe also shot much more 3-pointers. In that season, 42 percent of his shots are 3-pointers which is almost the double of average percentage of 3-pointers in his career before that season. 

In [ ]:
#Calculation of accuracy of each shot type
acc_shot_type = kobe_shotlog.groupby(['Season','SHOT_TYPE'])['SHOT_MADE_FLAG'].value_counts().to_frame()
acc_shot_type.rename(columns={'SHOT_MADE_FLAG':'Counts'},inplace=True)
acc_shot_type.reset_index(inplace=True)
acc_shot_type

#calculate the accuracy of each shot type(0 means missed, 1 means made)
acc_shot_type['Percentage'] = np.nan
for season in seasons:
    two_made = acc_shot_type.loc[(acc_shot_type['Season']==season) & (acc_shot_type['SHOT_TYPE']=='2PT Field Goal') & (acc_shot_type['SHOT_MADE_FLAG']==1)].Counts.to_numpy()[0]
    two_miss = acc_shot_type.loc[(acc_shot_type['Season']==season) & (acc_shot_type['SHOT_TYPE']=='2PT Field Goal') & (acc_shot_type['SHOT_MADE_FLAG']==0)].Counts.to_numpy()[0]
    three_made = acc_shot_type.loc[(acc_shot_type['Season']==season) & (acc_shot_type['SHOT_TYPE']=='3PT Field Goal') & (acc_shot_type['SHOT_MADE_FLAG']==1)].Counts.to_numpy()[0]
    three_miss = acc_shot_type.loc[(acc_shot_type['Season']==season) & (acc_shot_type['SHOT_TYPE']=='3PT Field Goal') & (acc_shot_type['SHOT_MADE_FLAG']==0)].Counts.to_numpy()[0]
    two_acc = round(two_made/(two_made+two_miss),3) * 100
    three_acc = round(three_made/(three_made+three_miss),3) * 100
    acc_shot_type.loc[(acc_shot_type['Season']==season) & (acc_shot_type['SHOT_TYPE']=='2PT Field Goal') & (acc_shot_type['SHOT_MADE_FLAG']==1),'Percentage'] = two_acc
    acc_shot_type.loc[(acc_shot_type['Season']==season) & (acc_shot_type['SHOT_TYPE']=='2PT Field Goal') & (acc_shot_type['SHOT_MADE_FLAG']==0),'Percentage'] = (100-two_acc)
    acc_shot_type.loc[(acc_shot_type['Season']==season) & (acc_shot_type['SHOT_TYPE']=='3PT Field Goal') & (acc_shot_type['SHOT_MADE_FLAG']==1),'Percentage'] = three_acc
    acc_shot_type.loc[(acc_shot_type['Season']==season) & (acc_shot_type['SHOT_TYPE']=='3PT Field Goal') & (acc_shot_type['SHOT_MADE_FLAG']==0),'Percentage'] = (100-three_acc)
#graph 2pt accuracy
#extract 2pt shot information
two = acc_shot_type[(acc_shot_type['SHOT_MADE_FLAG']==1) & (acc_shot_type['SHOT_TYPE']=='2PT Field Goal')]
three = acc_shot_type[(acc_shot_type['SHOT_MADE_FLAG']==1) & (acc_shot_type['SHOT_TYPE']=='3PT Field Goal')]
fig = go.Figure(data=[
    go.Bar(name='2PT%', x=two.Season, y=two.Percentage, marker_color='#fdb927'),
    go.Bar(name='3PT%', x=three.Season, y=three.Percentage, marker_color='#552583')
])#552583, fdb927
fig.update_layout(barmode='group', title='Field Goal Percentage by Shot Type', title_x=0.5, width=800, height=500)
fig.update_xaxes(type='category', title_text='Season')
fig.update_yaxes(title_text='Field  Goal  Percentage')
fig.show()

As we can see on the graph above, Kobe had been a good shooter inside of the 3-point line. Of all the 2-pointers he shot in his career, he made around 45% to 50% of them in the majority of his career. However, in his last two season, his performance was significantly impacted by the career ending injury and his 2-pointers percentage dropped by 5 percent. 

Kobe Bryant was not the most efficient 3PT shooter comparing to Stephen Curry or Klay Thompson. On a average, Kobe made around 33% of his 3PT shots. However, this percentage dropped a bit after he torn his achilles in the last few seasons. 


### Graph Kobe Bryant's shot chart
First I need to define a couple functions to draw plot Kobe's shot chart
1. Draw the court
2. Plot all Kobe's shots

In [ ]:
#Draw the NBA Court
def draw_court(ax=None, color='black', lw=2, outer_lines=False):
    #if ax object is not provided, get current one
    if ax is None:
        ax = plt.gca()
        
    #draw the court
    court_elements = []
    #create HOOP (has a radius of 9" which is 7.5 on our coordinate system)
    hoop = Circle((0,0), radius=7.5, linewidth=lw, color='red', fill=False)
    court_elements.append(hoop)
    
    #create BACKBOARD
    backboard = Rectangle((-30,-7.5), 60, -1, linewidth=lw, color=color)
    court_elements.append(backboard)
    
    #The PAINT
    #create outer box of the paint(width=16ft, height=19ft)
    outer_box = Rectangle((-80,-47.5), 160, 190, linewidth=lw, color=color, fill=False)
    court_elements.append(outer_box)
    #create inner box of the paint(width=12ft, height=19ft)
    inner_box = Rectangle((-60,-47.5), 120, 190, linewidth=lw, color=color, fill=False)
    court_elements.append(inner_box)
    
    #create FREETHROW TOP ARC
    top_free_throw = Arc((0,142.5), 120, 120, theta1=0, theta2=180, linewidth=lw, color=color, fill=False)
    court_elements.append(top_free_throw)
    #create FREETHROW BOTTOM ARCabs
    bottom_free_throw = Arc((0,142.5), 120, 120, theta1=180, theta2=0, linewidth=lw, color=color, linestyle="dashed")
    court_elements.append(bottom_free_throw)
    
    #create RESTRICTED ZONE
    restricted = Arc((0,0), 80, 80, theta1=0, theta2=180, linewidth=lw, color=color)
    court_elements.append(restricted)
    
    #THREE POINT LINE
    #left 3pt line(straight, 14ft long)
    left_three = Rectangle((-220,-47.5), 0, 140, linewidth=lw, color=color)
    court_elements.append(left_three)
    #right 3pt line(straight)
    right_three = Rectangle((220,-47.5), 0, 140, linewidth=lw, color=color)
    court_elements.append(right_three)
    #3pt arc
    three_arc = Arc((0,0), 475, 475, theta1=22, theta2=158, linewidth=lw, color=color)
    court_elements.append(three_arc)
    
    #create CENTRE ARC
    center_outer_arc = Arc((0, 422.5), 120, 120, theta1=180, theta2=0,
                           linewidth=lw, color=color)
    center_inner_arc = Arc((0, 422.5), 40, 40, theta1=180, theta2=0,
                           linewidth=lw, color=color)
    court_elements.append(center_outer_arc)
    court_elements.append(center_inner_arc)
    
    #BORDER LINES
    if outer_lines:
        outer_lines = Rectangle((-250,-47.5), 500, 470, linewidth=lw, color=color, fill=False)
        court_elements.append(outer_lines)
    
    #draw each court element on axes
    for element in court_elements:
        ax.add_patch(element)
        
    return ax

In [ ]:
#plot all the shots
def draw_shotchart(df, ax=None, title=""):
    
    plt.rcParams['figure.figsize'] = (13, 12)
    
    if ax is None:
        ax = plt.gca()
    
    ax.set_title(title, fontsize=18)
    #draw court
    draw_court(ax, outer_lines=True)
    ax.set_xlim(-250.5,250)
    ax.set_ylim(422.5,-47.5)
    
    #draw made shot
    x_made = df[df['EVENT_TYPE']=='Made Shot']['LOC_X']
    y_made = df[df['EVENT_TYPE']=='Made Shot']['LOC_Y']
    ax.scatter(x_made,y_made,facecolors='none', edgecolors='g', marker="o",s=50, linewidth=1.5)
   
    #draw missed shot
    x_missed = df[df['EVENT_TYPE']=='Missed Shot']['LOC_X']
    y_missed = df[df['EVENT_TYPE']=='Missed Shot']['LOC_Y']
    ax.scatter(x_missed, y_missed, c='r', marker='x', s=50, linewidth=0.5)
    
    return ax

In [ ]:
draw_shotchart(kobe_shotlog, title="Shot Chart of Kobe's Career")
plt.show()

This graph looks way too messy since it contain over 30000 shots Kobe took in his entire career. For easier understanding and better analysis of Kobe's shots, we will create a hexagon shot chart. 

In [ ]:
#draw hexagon shot chart
#create a hexbin graph and fit all the shots in each hexbin, use the data to plot the chart
def draw_hexbin_chart(df,filter_threshold, title=""):
    np.seterr(divide='ignore', invalid='ignore')
    #calculate the accuracy of shots for each hexbin
    #get hexbins of all shots
    shots_hex = plt.hexbin(x=df.LOC_X, y=df.LOC_Y, extent=(-250, 250, 422.5, -47.5), cmap='Blues', gridsize=35)
    plt.close()
    #get hexbins of all made shots
    made_shots = df[df['EVENT_TYPE']=='Made Shot']
    made_hex = plt.hexbin(x=made_shots.LOC_X, y=made_shots.LOC_Y,extent=(-250, 250, 422.5, -47.5), cmap='Blues', gridsize=35)
    plt.close()
    acc_hex = made_hex.get_array() / shots_hex.get_array()
    acc_hex[np.isnan(acc_hex)] = 0 

    #filter the shots
    sample_sizes = shots_hex.get_array()
    #Get rid of the bins that has less than 20 shots
    filter_threshold = filter_threshold
    for i in range(len(acc_hex)):
        if sample_sizes[i] < filter_threshold:
            acc_hex[i] = 0
    x = [i[0] for i in shots_hex.get_offsets()]
    y = [i[1] for i in shots_hex.get_offsets()]
    z = acc_hex
    
    #calculate frequency or size of each hexbin
    test = plt.hexbin(x=df.LOC_X, y=df.LOC_Y, extent=(-250, 250, 422.5, -47.5), cmap='Blues', gridsize=35)
    freq_hex = test.get_array() / sum(test.get_array())
    #fix the error that results in extremely big frequency in the cell near the hoop
    hexcell = df.loc[(df.LOC_X > 0) & (df.LOC_X < 14) & (df.LOC_Y>-8) & (df.LOC_Y<8)]
    #calculate frequency and accuracy of that cell
    cell_made = hexcell[hexcell['SHOT_MADE_FLAG']==1]
    if len(hexcell)==0:
        cell_acc=0
    else:
        cell_acc = len(cell_made) / len(hexcell)
    cell_freq = len(hexcell) / sum(test.get_array())
    index = np.where(freq_hex == max(freq_hex))
    np.put(freq_hex, index, cell_freq)
    np.put(acc_hex, index, cell_acc)
    sizes = freq_hex / max(freq_hex) * 500
    plt.close()
    
    fig = plt.figure(figsize=(13, 12))
    plt.xlim(-250,250)
    plt.ylim(422.5, -47.5)
    plt.title(title, fontdict = {'fontsize' : 25})
    
    #plot the hexbins
    scatter = plt.scatter(x, y, c=z, s=sizes, cmap=plt.cm.YlOrRd, marker='h')
    draw_court(outer_lines=True)
    
    cur_axes = plt.gca()
    cur_axes.axes.get_xaxis().set_visible(False)
    cur_axes.axes.get_yaxis().set_visible(False)

    legend1 = plt.legend(
        *scatter.legend_elements(num=6, fmt="{x:.0f}%", func=lambda s: s * 100),
        loc="lower right", title='Shot\nAccuracy', fontsize=15)
    legend1.get_title().set_fontsize('15')
    
    plt.gca().add_artist(legend1)
    

In [ ]:
draw_hexbin_chart(kobe_shotlog, 35, "Shot Chart of Kobe's Career")

(The larger the hexagon is, the more shots was taken in that location. And the darker the hexagon is, the higer the FG%.)
<br>

As we can see on Kobe's shot chart, Kobe's hot zone covered most area on the court and he was able to make shots anywhere he wants. Kobe's FG% was highest under the rim and it was the lowest near the corner of the court. More generally, Kobe shot a bit more on the right side of the court and made a bit more on the right side since right hand was his dominant hand. 

## 8 VS 24

In the 20 seasons of Kobe's career, he wore the No.8 jersey for the first 10 years and wore the No.24 jersey for the last 10 years. According to him, these two numbers represented two different time periods of his career both on and off the court where No.8 shows his "nonstop energy and aggresiveness" and No.24 shows his "growth and maturity". 

In [ ]:
seasons_8 = ['1996-97', '1997-98', '1998-99', '1999-00', '2000-01', '2001-02', '2002-03', '2003-04','2004-05', '2005-06']
shots_8 = kobe_shotlog[(kobe_shotlog['Season'].isin(seasons_8)) & (kobe_shotlog.SHOT_DISTANCE < 40)].copy()
shots_24 = kobe_shotlog[(~ kobe_shotlog['Season'].isin(seasons_8)) & (kobe_shotlog.SHOT_DISTANCE < 40)].copy()
shots_8.SHOT_DISTANCE=pd.to_numeric(shots_8.SHOT_DISTANCE)
shots_24.SHOT_DISTANCE=pd.to_numeric(shots_24.SHOT_DISTANCE)
shots_8.LOC_X=pd.to_numeric(shots_8.LOC_X)
shots_8.LOC_Y=pd.to_numeric(shots_8.LOC_Y)
shots_24.LOC_X=pd.to_numeric(shots_24.LOC_X)
shots_24.LOC_Y=pd.to_numeric(shots_24.LOC_Y)

In [ ]:
plt.figure(figsize=(15,9))
sns.kdeplot(data=shots_24['SHOT_DISTANCE'], label='No.24', shade=True)
sns.kdeplot(data=shots_8['SHOT_DISTANCE'],label='No.8', shade=True)
plt.legend(fontsize=20)

According to the graph above, the two curves have similar patterns which indicates that Kobe's shot distance in the different jerseys are similar or the his range did not change. The majority of the shots Kobe are shown as the three peaks on the graph which are under the basket, 16-19 feet away from basket(long range jumpers), and 24-25 feet away from the basket(3-pointers). 

However, there are two major discrepancies between the two curves. The curve of Kobe with No.8 has a much higher peak under the basket and a lower peak around the 3PT line. Such discrepancies illustrates that Kobe, when wearing No.8, tended to take more shots under the basket. And when wearing No.24, he tended to take more shots around the 3PT line comparing to the previous 10 seasons.

In [ ]:
def draw_jointplot(df):
    cmap=plt.cm.gnuplot_r
    
    joint_shot_chart = sns.jointplot(x=df.LOC_X, y=df.LOC_Y,
                                 kind='kde', space=0, color=cmap(0.1),
                                 cmap=cmap, n_levels=50, shade=True)

    joint_shot_chart.fig.set_size_inches(12,11)

    # A joint plot has 3 Axes, the first one called ax_joint 
    # is the one we want to draw our court onto and adjust some other settings
    ax = joint_shot_chart.ax_joint
    draw_court(ax)

    # Adjust the axis limits and orientation of the plot in order
    # to plot half court, with the hoop by the top of the plot
    ax.set_xlim(-250,250)
    ax.set_ylim(422.5, -47.5)

    # Get rid of axis labels and tick marks
    ax.set_xlabel('')
    ax.set_ylabel('')
    ax.tick_params(labelbottom='off', labelleft='off')

### Kobe Bryant's Shot Chart heatmap when wearing No.8

In [ ]:
draw_jointplot(shots_8)

### Kobe Bryant's Shot Chart heatmap when wearing No.24

In [ ]:
draw_jointplot(shots_24)

Similar patterns can be observed with the heat map above. In the graphs above, the darker the area, the more shots were taken in that area. And we can see that in the 2nd graph(No.24), the color of the Wing and the right side of the extended free-throw line is much darker than that of the the 1st graph(No.8) which also shows that Kobe took more long range shots when wearing No.24. 


This may be a result of, according to Kobe Bryant, "weakened" physical attributes. Although Kobe is still one of the most athletic players in the league, he is not as dominant and flexible under the basket as he used to be in the No.8 Jersey. Thus he shifted his gamestyle and started to take more longer range shots and thrived with this new style of play.  